In [1]:
import pandas as pd
import numpy as np
import logging
from typing import List, Tuple

In [2]:

# Configure logging for model validation trails
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class OptimalRiskQuantizer:
    """
    A quantitative tool to optimally bucket continuous risk variables 
    by maximizing the Log-Likelihood of default probabilities using Dynamic Programming.
    """
    
    def __init__(self, num_buckets: int = 10, min_bucket_size: int = 100):
        self.num_buckets = num_buckets
        self.min_bucket_size = min_bucket_size
        self.boundaries_: List[float] = []
        self.bucket_stats_: pd.DataFrame = pd.DataFrame()
        
    def _compute_log_likelihood(self, n_total: int, k_defaults: int) -> float:
        """
        Calculates the log-likelihood of a specific bucket.
        Formula: K * ln(p) + (N - K) * ln(1 - p)
        """
        if n_total == 0:
            return 0.0
        
        p = k_defaults / n_total
        
        # Handle edge cases to prevent log(0) domain errors
        if p == 0.0 or p == 1.0:
            return 0.0
            
        return k_defaults * np.log(p) + (n_total - k_defaults) * np.log(1 - p)

    def fit(self, X: pd.Series, y: pd.Series) -> 'OptimalRiskQuantizer':
        """
        Executes the DP algorithm to find optimal boundaries.
        
        Parameters:
        X (pd.Series): The continuous feature (e.g., FICO scores)
        y (pd.Series): The binary target (0 = No Default, 1 = Default)
        """
        logging.info(f"Starting DP Optimization for {self.num_buckets} buckets...")
        
        # 1. Aggregate the data for computational efficiency
        df = pd.DataFrame({'feature': X, 'target': y})
        grouped = df.groupby('feature')['target'].agg(n='count', k='sum').reset_index()
        grouped = grouped.sort_values('feature').reset_index(drop=True)
        
        num_unique = len(grouped)
        
        # 2. Precompute cumulative arrays for O(1) range queries
        cum_n = np.zeros(num_unique + 1, dtype=int)
        cum_k = np.zeros(num_unique + 1, dtype=int)
        
        for i in range(num_unique):
            cum_n[i+1] = cum_n[i] + grouped['n'].iloc[i]
            cum_k[i+1] = cum_k[i] + grouped['k'].iloc[i]
            
        # 3. Initialize DP Matrices
        # dp[b][i] stores the max log-likelihood using 'b' buckets for the first 'i' unique values
        dp = np.full((self.num_buckets + 1, num_unique + 1), -np.inf)
        dp[0][0] = 0.0
        pointers = np.zeros((self.num_buckets + 1, num_unique + 1), dtype=int)
        
        # 4. DP Optimization Matrix Traversal
        for b in range(1, self.num_buckets + 1):
            for i in range(1, num_unique + 1):
                for j in range(i):
                    n_bucket = cum_n[i] - cum_n[j]
                    k_bucket = cum_k[i] - cum_k[j]
                    
                    # Enforce the minimum bucket size risk constraint
                    if n_bucket >= self.min_bucket_size:
                        if dp[b-1][j] != -np.inf:
                            ll = self._compute_log_likelihood(n_bucket, k_bucket)
                            current_val = dp[b-1][j] + ll
                            
                            if current_val > dp[b][i]:
                                dp[b][i] = current_val
                                pointers[b][i] = j
                                
        # 5. Backtrack to reconstruct the optimal boundaries
        boundaries_idx = []
        curr_i = num_unique
        for b in range(self.num_buckets, 0, -1):
            if curr_i == 0:
                break
            curr_i = pointers[b][curr_i]
            boundaries_idx.append(curr_i)
            
        boundaries_idx = boundaries_idx[::-1]
        
        # Extract the actual feature values
        self.boundaries_ = [grouped['feature'].iloc[idx] for idx in boundaries_idx if idx < num_unique]
        logging.info("Optimization complete. Boundaries mapped.")
        
        # 6. Generate the Risk Profile Summary
        self._generate_summary(grouped)
        
        return self

    def _generate_summary(self, grouped_df: pd.DataFrame):
        """Builds a business-ready summary table of the generated buckets."""
        results = []
        prev_boundary = grouped_df['feature'].min()
        all_boundaries = self.boundaries_ + [grouped_df['feature'].max() + 1] # Add +1 to catch the max value
        
        for i, b in enumerate(all_boundaries):
            mask = (grouped_df['feature'] >= prev_boundary) & (grouped_df['feature'] < b)
            subset = grouped_df[mask]
            
            n_total = subset['n'].sum()
            k_defaults = subset['k'].sum()
            pd_rate = k_defaults / n_total if n_total > 0 else 0
            
            results.append({
                'Bucket': i + 1,
                'Min_FICO': prev_boundary,
                'Max_FICO': b - 1 if b <= grouped_df['feature'].max() else grouped_df['feature'].max(),
                'Total_Loans': n_total,
                'Defaults': k_defaults,
                'Probability_of_Default (PD)': round(pd_rate, 4)
            })
            prev_boundary = b
            
        self.bucket_stats_ = pd.DataFrame(results)



# ==========================================
# EXECUTION & REPORTING
# ==========================================

In [3]:

if __name__ == "__main__":
    # 1. Load the data
    df = pd.read_csv('Task 3 and 4_Loan_Data.csv')
    
    # 2. Initialize and fit the Quantizer
    quantizer = OptimalRiskQuantizer(num_buckets=10, min_bucket_size=100)
    quantizer.fit(df['fico_score'], df['default'])
    
    # 3. Output results for Charlie and the Risk Team
    print("\n--- Optimal FICO Score Boundaries ---")
    print([int(b) for b in quantizer.boundaries_])
    
    print("\n--- Risk Bucket Profile (For Model Validation) ---")
    # Formatting to percentages for easier stakeholder reading
    formatted_stats = quantizer.bucket_stats_.copy()
    formatted_stats['Probability_of_Default (PD)'] = (formatted_stats['Probability_of_Default (PD)'] * 100).astype(str) + '%'
    print(formatted_stats.to_string(index=False))

INFO: Starting DP Optimization for 10 buckets...
INFO: Optimization complete. Boundaries mapped.



--- Optimal FICO Score Boundaries ---
[408, 503, 521, 553, 581, 612, 650, 697, 733, 740]

--- Risk Bucket Profile (For Model Validation) ---
 Bucket  Min_FICO  Max_FICO  Total_Loans  Defaults Probability_of_Default (PD)
      1       408       407            0         0                        0.0%
      2       408       502          168       121                      72.02%
      3       503       520          133        78         58.650000000000006%
      4       521       552          496       229                      46.17%
      5       553       580          911       307                       33.7%
      6       581       611         1561       381                      24.41%
      7       612       649         2465       402                      16.31%
      8       650       696         2609       256                       9.81%
      9       697       732         1104        64          5.800000000000001%
     10       733       739          122         0                  